# 🎬 PHA 2: DEEP LEARNING & PHỄU LỌC 3 TẦNG SONG SONG TRÊN 2x GPU TESLA T4
## Kiến Trúc Video-Grouped Frame-Pool & Tăng Tốc Siêu Tốc (Zero-Disk Overflow & Zero-Stderr Warnings)
**Đồ án PBL6 - Hệ Thống Nội Suy Khung Hình Anime Bằng Thin-Plate Spline & Deep Learning**

---

### ⚡ CÁC ĐẶC ĐIỂM & NÂNG CẤP TỐI ƯU CỰC ĐẠI:
1. **Triệt Tiêu 100% Cảnh Báo Màu Đỏ (`sys.stderr`):**
   - Khóa toàn bộ warning bằng `warnings.filterwarnings("ignore")` và `PYTHONWARNINGS="ignore"`.
   - Sử dụng chuẩn `torch.amp.autocast('cuda')` hiện đại, triệt tiêu hoàn toàn `FutureWarning` của PyTorch.
   - Thiết lập `torch.cuda.set_device(worker.device)` ngay đầu mỗi worker thread để cô lập tài nguyên GPU chuẩn xác.
   - Định hướng thanh tiến độ `tqdm(file=sys.stdout)` sang luồng stdout để loại bỏ hộp thoại đỏ trên Kaggle.
   - Thiết lập `OPENCV_LOG_LEVEL="OFF"` tắt toàn bộ thông báo phụ của OpenCV C++.
2. **Tăng Tốc Đột Phá GPU/CPU (Giải Quyết Nghẽn Cổ Chai CPU & Bộ Nhớ):**
   - **Full-GPU Tensor Batching (Tách Nền & Purity):** Đưa toàn bộ ảnh `uint8` thẳng lên GPU; thực hiện resize, pad, sigmoid, cắt padding, tính `fg_ratios`, trộn nền trắng và tính Canvas Purity (`pwcr`, `rgd`, `density`) 100% trên PyTorch CUDA $\implies$ Loại bỏ hoàn toàn ép kiểu `float64` và vòng lặp NumPy trên CPU.
   - **Tách Rời GlueStick Feature Extraction & Matching:** Trích xuất đặc trưng GlueStick theo từng frame đơn lẻ (chạy SuperPoint & LSD đúng 1 lần duy nhất cho mỗi frame) và lưu vào cache; ghép cặp matching trực tiếp trên GPU với AMP $\implies$ Triệt tiêu hoàn toàn việc chạy lại `cv2_lsd` trên CPU nhiều lần cho cùng 1 frame.
   - **SSPR Tier 2.5 Early-Exit (0.2 ms):** Kiểm tra tỷ lệ nét tĩnh trùng khít giữa 3 frame `calc_sspr_fast` trước khi gọi GlueStick matching và Optical Flow $\implies$ Loại bỏ ngay các triplet đứng im / lặp frame mà không tốn chi phí inference.
   - **Fast Optical Flow với 2 Tầng Early-Exit & Magnitude Cache:** Farneback ở độ phân giải $1/2$ kèm magnitude được tính và cache đồng thời; nếu chặng 1 đứng yên thì bỏ qua luôn chặng 2; chỉ chạy backward warping `cv2.remap` (với grid `float32` cache sẵn) khi vượt qua các ngưỡng động học.
   - **Fast PNG Staging Writes:** Sử dụng `cv2.IMWRITE_PNG_COMPRESSION, 1` khi ghi ảnh tạm trước khi nén ZIP $\implies$ Tăng tốc độ ghi đĩa staging gấp 4–5 lần.
3. **Phễu Lọc 3 Tầng Chuẩn Hóa & Triệt Tiêu False Positives:**
   - **Gate 0:** GlueStick bắt điểm chuẩn xác trên Foreground nhân vật + Kiểm tra tính liên tục qua frame giữa `imggt` ($\ge 60\%$).
   - **Gate 1:** Canvas Purity đo trên ảnh Lineart + Kiểm tra tỷ lệ nhân vật $0.03 \le \text{fg\_ratio} \le 0.85$ + Nới $\text{BSPI} \le 0.15$.
   - **Gate 2:** Stroke Topology áp dụng Dilation liên thông nét nhân vật ($\text{DCCR} \ge 0.55$, $\text{MCCC} \le 250$).
   - **Gate 3:** Farneback Optical Flow trên ảnh nhân vật có gradient mịn + Sửa đúng chiều backward warping `cv2.remap` ($\text{WSDS} \ge 0.18$).
4. **Quy Trình Chạy Thủ Công Từng Cell & Dọn Sạch Đĩa (Run Cell-by-Cell & Zero-Disk Overflow):**
   - **Bước 8:** Xuất Part 1 ZIP (`sakuga_golden_dataset_tps_part1.zip`) và lưu `golden_meta_part1.json`.
   - **Bước 10:** Dọn đĩa giải phóng 100% dung lượng.
   - **Bước 10.5 (Mới):** Tự động khôi phục metadata Part 1 từ `/kaggle/input` hoặc `/kaggle/working` nếu restart session.
   - **Bước 11:** Xuất Part 2 ZIP (`sakuga_golden_dataset_tps_part2.zip`), gộp `triplets_combined.csv` và `quality_audit_report_combined.csv`.
5. **Đánh Số Clip Chuẩn Xác & Không Xung Đột (Canonical ID Continuity):**
   - Part 1: `clip_000001` $\to$ `clip_XXXXXX`. Part 2: `clip_XXXXXX+1` $\to$ `clip_YYYYYY`.


---
### 🛠️ Bước 1: Kiểm Tra 2x GPU & Cài Đặt Môi Trường Thư Viện Deep Learning


In [ ]:
import os
import sys

# Khóa toàn bộ cảnh báo Python, PyTorch & OpenCV để triệt tiêu log đỏ trên giao diện Kaggle
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["OPENCV_LOG_LEVEL"] = "OFF"
import warnings
warnings.filterwarnings("ignore")

!nvidia-smi

# Cài đặt các thư viện cần thiết tương thích với PyTorch & CUDA trên Kaggle
!pip install -q timm gdown einops kornia omegaconf

import glob
import json
import time
import shutil
import random
import zipfile
import gc
import csv
import types
from collections import defaultdict
import cv2

# Khóa log OpenCV C++ an toàn để tránh in cảnh báo ra stderr trên mọi phiên bản OpenCV
try:
    if hasattr(cv2, 'utils') and hasattr(cv2.utils, 'logging'):
        cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_SILENT)
    elif hasattr(cv2, 'setLogLevel'):
        cv2.setLogLevel(0)
except Exception:
    pass

# Giới hạn OpenCV 1 thread cho mỗi worker để chống nghẽn CPU & deadlock OpenMP khi chạy song song 2 GPU
cv2.setNumThreads(1)

import torch
import torch.nn.functional as F
import numpy as np
from scipy import ndimage
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

print("=" * 75)
print(f"[*] Python Version  : {sys.version.split()[0]}")
print(f"[*] PyTorch Version : {torch.__version__}")
print(f"[*] OpenCV Version  : {cv2.__version__}")
num_gpus = torch.cuda.device_count()
print(f"[*] Số lượng GPU    : {num_gpus}")
if torch.cuda.is_available():
    for i in range(num_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"    + GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
print("=" * 75)

# Clone các repository mã nguồn Deep Learning
if not os.path.exists("GlueStick"):
    !git clone --depth 1 https://github.com/cvg/GlueStick.git
    !cd GlueStick && pip install -q --no-deps -e . && cd ..

if not os.path.exists("AniLines"):
    !git clone --depth 1 https://github.com/zhenglinpan/AniLines-Anime-Lineart-Extractor.git AniLines

if not os.path.exists("anime_seg"):
    !git clone --depth 1 https://github.com/SkyTNT/anime-segmentation.git anime_seg

for p in ["GlueStick", "AniLines", "anime_seg", "anime_seg/model"]:
    p_abs = os.path.abspath(p)
    if p_abs not in sys.path:
        sys.path.insert(0, p_abs)

# Mock pytlsd bằng OpenCV LSD native C++ an toàn, chống rò rỉ RAM và chống vô tận vòng lặp CPU
def cv2_lsd(img, scale=1.0):
    if not isinstance(img, np.ndarray):
        img = np.array(img)
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if img.dtype != np.uint8:
        if img.max() <= 1.01 and img.max() > 0:
            img = (img * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img = np.clip(img, 0, 255).astype(np.uint8)
    if scale != 1.0:
        h, w = img.shape[:2]
        img_s = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    else:
        img_s = img
    detector = cv2.createLineSegmentDetector(cv2.LSD_REFINE_STD)
    lines, _, _, _ = detector.detect(img_s)
    if lines is None or len(lines) == 0:
        return np.zeros((0, 5), dtype=np.float32)
    lines = lines.reshape(-1, 4).astype(np.float32)
    if scale != 1.0:
        lines /= float(scale)
    scores = np.ones((len(lines), 1), dtype=np.float32)
    return np.hstack([lines, scores])

fake_pytlsd = types.ModuleType("pytlsd")
fake_pytlsd.lsd = cv2_lsd
sys.modules["pytlsd"] = fake_pytlsd

try:
    import gluestick.models.wireframe as wireframe_mod
    wireframe_mod.lsd = cv2_lsd
except Exception:
    pass

print("[✓] Môi trường và cơ chế Mock LSD chống rò rỉ RAM đã sẵn sàng!")


---
### 📥 Bước 2: Tự Động Tải Trọng Số Pretrained (GlueStick, ISNetDIS, AniLines Detail)


In [ ]:
os.makedirs("weights", exist_ok=True)

# 1. Trọng số GlueStick MD Checkpoint
gluestick_ckpt = "weights/checkpoint_GlueStick_MD.tar"
if not os.path.exists(gluestick_ckpt) or os.path.getsize(gluestick_ckpt) < 1000000:
    print("[*] Đang tải GlueStick checkpoint...")
    url_gs = "https://github.com/cvg/GlueStick/releases/download/v0.1_arxiv/checkpoint_GlueStick_MD.tar"
    !wget -q -O {gluestick_ckpt} {url_gs}
    print("[✓] GlueStick weights tải xong!")

# 2. Trọng số ISNetDIS (Anime Background Segmentation)
isnet_ckpt = "weights/isnetis.ckpt"
if not os.path.exists(isnet_ckpt) or os.path.getsize(isnet_ckpt) < 1000000:
    print("[*] Đang tải ISNetDIS checkpoint...")
    url_isnet = "https://huggingface.co/skytnt/anime-seg/resolve/main/isnetis.ckpt"
    !wget -q -O {isnet_ckpt} {url_isnet}
    print("[✓] ISNetDIS weights tải xong!")

# 3. Trọng số AniLines Detail
anilines_ckpt = "weights/detail.pth"
if not os.path.exists(anilines_ckpt) or os.path.getsize(anilines_ckpt) < 1000000:
    print("[*] Đang tải AniLines Detail checkpoint...")
    !gdown --id 12U1Mwlonoipk2Yvr12mNaFB30foy420o -O {anilines_ckpt} --fuzzy
    print("[✓] AniLines Detail weights tải xong!")

print()
print("=" * 60)
print("[✓] Danh mục trọng số Deep Learning trong thư mục weights/:")
!ls -lh weights/
print("=" * 60)


---
### 🧠 Bước 3: Khởi Tạo Lớp Dual-GPU Pipeline Worker (Hỗ Trợ 2x GPU Song Song & GPU Batching)


In [ ]:
from gluestick.models.two_view_pipeline import TwoViewPipeline
from network.line_extractor import LineExtractor

try:
    from anime_seg.model.isnet import ISNetDIS
except ImportError:
    try:
        from model.isnet import ISNetDIS
    except ImportError:
        from isnet import ISNetDIS

def load_gluestick(device, weights_path="weights/checkpoint_GlueStick_MD.tar"):
    conf = {
        'name': 'two_view_pipeline',
        'use_lines': True,
        'extractor': {
            'name': 'wireframe',
            'sp_params': {'force_num_keypoints': False, 'max_num_keypoints': 1000},
            'wireframe_params': {'merge_points': True, 'merge_line_endpoints': True},
            'max_n_lines': 300,
        },
        'matcher': {'name': 'gluestick', 'weights': weights_path, 'trainable': False},
        'ground_truth': {'from_pose_depth': False}
    }
    model = TwoViewPipeline(conf).to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model

class AnimeBgRemover:
    def __init__(self, device, ckpt_path="weights/isnetis.ckpt", img_size=1024):
        self.device = device
        self.img_size = img_size
        self.net = ISNetDIS().to(self.device)
        try:
            ckpt = torch.load(ckpt_path, map_location=self.device, weights_only=False)
        except TypeError:
            ckpt = torch.load(ckpt_path, map_location=self.device)
        sd = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
        clean_sd = {k.replace('net.', ''): v for k, v in sd.items() if k.startswith('net.')}
        if not clean_sd:
            clean_sd = sd
        self.net.load_state_dict(clean_sd)
        self.net.eval()
        for p in self.net.parameters():
            p.requires_grad = False

    def remove_background_batch(self, img_bgr_list, batch_size=16):
        """
        Xử lý tách nền theo Batch trên GPU cho danh sách ảnh BGR 540p (batch_size=16).
        Tối ưu hóa:
        - Chuyển toàn bộ ảnh uint8 lên GPU, float/255.0 và resize + pad trực tiếp trên GPU.
        - Trộn nền trắng và tính fg_ratios trực tiếp trên GPU bằng PyTorch tensor.
        - Trả về danh sách: [(white_bg_uint8, fg_mask_bool, float(fg_ratio)), ...]
        """
        results = []
        n_imgs = len(img_bgr_list)
        if n_imgs == 0:
            return results

        s = self.img_size
        for start_idx in range(0, n_imgs, batch_size):
            sub_imgs = img_bgr_list[start_idx : start_idx + batch_size]
            b_size = len(sub_imgs)

            h0, w0 = sub_imgs[0].shape[:2]
            h, w = (s, int(s * w0 / h0)) if h0 > w0 else (int(s * h0 / w0), s)
            ph, pw = s - h, s - w
            pad_left = pw // 2
            pad_right = pw - pad_left
            pad_top = ph // 2
            pad_bottom = ph - pad_top

            # Chuyển batch uint8 lên GPU -> permute -> float / 255.0 trên GPU
            stack_u8 = np.stack(sub_imgs, axis=0)
            orig_t = torch.from_numpy(stack_u8).to(self.device, non_blocking=True).permute(0, 3, 1, 2).float() / 255.0

            # Resize & Pad trực tiếp trên GPU
            resized_t = F.interpolate(orig_t, size=(h, w), mode='bilinear', align_corners=False)
            inp_tensor = F.pad(resized_t, (pad_left, pad_right, pad_top, pad_bottom))

            with torch.no_grad():
                if 'cuda' in self.device.type:
                    with torch.amp.autocast('cuda'):
                        out = self.net(inp_tensor)
                else:
                    out = self.net(inp_tensor)

                # ISNetDIS trả về tuple ([d1, d2, d3, d4, d5, d6], [features...])
                if isinstance(out, (list, tuple)):
                    first = out[0]
                    raw_pred = first[0] if isinstance(first, (list, tuple)) else first
                else:
                    raw_pred = out

                pred = raw_pred.sigmoid() if hasattr(raw_pred, 'sigmoid') else torch.sigmoid(raw_pred)

                if pred.ndim == 3:
                    pred = pred.unsqueeze(1)

                pred_crop = pred[:, :, pad_top : pad_top + h, pad_left : pad_left + w]
                masks_t = F.interpolate(pred_crop.float(), size=(h0, w0), mode='bilinear', align_corners=False)

                # Tính fg_ratios trực tiếp trên GPU
                fg_ratios = (masks_t > 0.5).float().mean(dim=(1, 2, 3)).cpu().numpy()

                # Trộn nền trắng trực tiếp trên GPU
                white_t = torch.clamp(masks_t * orig_t * 255.0 + (1.0 - masks_t) * 255.0, 0, 255).to(torch.uint8)
                white_bgr_np = white_t.permute(0, 2, 3, 1).contiguous().cpu().numpy()
                fg_masks_np = (masks_t.squeeze(1) > 0.5).cpu().numpy()

            for b_i in range(b_size):
                results.append((white_bgr_np[b_i], fg_masks_np[b_i], float(fg_ratios[b_i])))

            del orig_t, resized_t, inp_tensor, out, raw_pred, pred, pred_crop, masks_t, white_t

        return results

    def remove_background(self, img_bgr):
        return self.remove_background_batch([img_bgr], batch_size=1)[0][0]

def load_line_extractor(device, weights_path="weights/detail.pth"):
    extractor = LineExtractor(2, 1, True).to(device)
    try:
        sd_line = torch.load(weights_path, map_location=device, weights_only=True)
    except TypeError:
        sd_line = torch.load(weights_path, map_location=device)
    extractor.load_state_dict(sd_line)
    extractor.eval()
    for p in extractor.parameters():
        p.requires_grad = False
    return extractor

# Lớp Worker đại diện cho 1 GPU cụ thể (chứa trọn gói 3 model trên GPU đó)
class DualGpuPipelineWorker:
    def __init__(self, device_id):
        device_str = f"cuda:{device_id}" if torch.cuda.is_available() else "cpu"
        self.device = torch.device(device_str)
        self.device_id = device_id
        print(f"[*] Đang khởi tạo DualGpuPipelineWorker trên: {self.device}...")
        self.gluestick = load_gluestick(self.device)
        self.bg_remover = AnimeBgRemover(self.device)
        self.line_extractor = load_line_extractor(self.device)

        # Sobel kernels trên GPU cho AniLines input
        sobel_x = torch.tensor([[-1.0, 0.0, 1.0],
                                [-2.0, 0.0, 2.0],
                                [-1.0, 0.0, 1.0]], dtype=torch.float32, device=self.device).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1.0, -2.0, -1.0],
                                [ 0.0,  0.0,  0.0],
                                [ 1.0,  2.0,  1.0]], dtype=torch.float32, device=self.device).view(1, 1, 3, 3)
        self.sobel_x = sobel_x
        self.sobel_y = sobel_y
        self.bgr_weights = torch.tensor([0.114, 0.587, 0.299], dtype=torch.float32, device=self.device).view(1, 3, 1, 1)

        print(f"[✓] Worker trên {self.device} đã sẵn sàng!")

    @torch.no_grad()
    def extract_lineart_and_edt_batch(self, white_imgs_list, batch_size=16):
        """
        Trích xuất Lineart bằng AniLines theo Batch trên GPU (batch_size=16),
        làm mịn nét trực tiếp trên CUDA, tính EDT (swdr) tức thì qua F.max_pool2d trên GPU,
        và tích hợp tính luôn Canvas Purity (pwcr, rgd, density) trực tiếp trên GPU.
        Trả về: [(line_uint8, float(swdr), {'pwcr': float, 'rgd': float, 'density': float}), ...]
        """
        results = []
        n_imgs = len(white_imgs_list)
        if n_imgs == 0:
            return results

        for start_idx in range(0, n_imgs, batch_size):
            sub_imgs = white_imgs_list[start_idx : start_idx + batch_size]
            b_size = len(sub_imgs)
            h0, w0 = sub_imgs[0].shape[:2]

            # [B, H, W, 3] uint8 -> [B, 3, H, W] float [0, 1] trên GPU
            stack_np = np.stack(sub_imgs, axis=0)
            t_bgr = torch.from_numpy(stack_np).to(self.device, non_blocking=True).permute(0, 3, 1, 2).float() / 255.0

            # Grayscale trên GPU
            grays = torch.sum(t_bgr * self.bgr_weights, dim=1, keepdim=True)

            # Sobel magnitude trên GPU
            pad = F.pad(grays, (1, 1, 1, 1), mode='replicate')
            gx = F.conv2d(pad, self.sobel_x)
            gy = F.conv2d(pad, self.sobel_y)
            mag = torch.sqrt(gx * gx + gy * gy + 1e-8)

            mag_min = mag.amin(dim=(2, 3), keepdim=True)
            mag_max = mag.amax(dim=(2, 3), keepdim=True)
            sobel_norm = 1.0 - (mag - mag_min) / (mag_max - mag_min + 1e-6)

            # Input vào AniLines: [B, 2, H, W]
            x_in = torch.cat([grays, sobel_norm], dim=1)

            pad_h = 8 - (h0 % 8) if (h0 % 8) != 0 else 0
            pad_w = 8 - (w0 % 8) if (w0 % 8) != 0 else 0
            if pad_h > 0 or pad_w > 0:
                x_in = F.pad(x_in, (0, pad_w, 0, pad_h), mode='reflect')

            # Bọc lời gọi self.line_extractor trong torch.amp.autocast('cuda') và ép .float()
            if 'cuda' in self.device.type:
                with torch.amp.autocast('cuda'):
                    pred_line = self.line_extractor(x_in)[:, :, :h0, :w0].float()
            else:
                pred_line = self.line_extractor(x_in)[:, :, :h0, :w0].float()

            # Làm mịn nét trên GPU (GPU Tensor Smoothing)
            try:
                import kornia.filters as kf
                pred_smooth = kf.bilateral_blur(pred_line, (5, 5), 0.15, (1.5, 1.5))
            except Exception:
                g_kernel = torch.tensor([[1., 2., 1.],
                                         [2., 4., 2.],
                                         [1., 2., 1.]], dtype=torch.float32, device=self.device).view(1, 1, 3, 3) / 16.0
                pad_line = F.pad(pred_line, (1, 1, 1, 1), mode='replicate')
                pred_smooth = F.conv2d(pad_line, g_kernel)

            # TÍNH TOÁN EDT (swdr) TỨC THÌ TRÊN GPU BẰNG F.max_pool2d
            stroke_mask = (pred_smooth < (180.0 / 255.0)).float()
            inv_stroke = 1.0 - stroke_mask
            pool_inv = F.max_pool2d(inv_stroke, kernel_size=7, stride=1, padding=3)
            thick_mask = (stroke_mask == 1.0) & (pool_inv == 0.0)

            total_strokes = stroke_mask.sum(dim=(2, 3)).squeeze(1).cpu().numpy()
            thick_strokes = thick_mask.sum(dim=(2, 3)).squeeze(1).cpu().numpy()

            # TÍNH TOÁN CANVAS PURITY TRỰC TIẾP TRÊN GPU TỪ pred_smooth
            lines_u8 = torch.clamp(pred_smooth.squeeze(1) * 255.0, 0, 255).round()
            pwcr_arr = (lines_u8 >= 250).float().mean(dim=(1, 2)).cpu().numpy()
            rgd_arr = ((lines_u8 >= 180) & (lines_u8 < 250)).float().mean(dim=(1, 2)).cpu().numpy()
            density_arr = (lines_u8 < 180).float().mean(dim=(1, 2)).cpu().numpy()

            lines_uint8 = lines_u8.to(torch.uint8).cpu().numpy()

            for b_i in range(b_size):
                tot_s = float(total_strokes[b_i])
                thk_s = float(thick_strokes[b_i])
                swdr = (thk_s / max(tot_s, 1.0)) if tot_s > 0 else 0.0
                cp_dict = {
                    'pwcr': float(pwcr_arr[b_i]),
                    'rgd': float(rgd_arr[b_i]),
                    'density': float(density_arr[b_i])
                }
                results.append((lines_uint8[b_i], float(swdr), cp_dict))

            del t_bgr, grays, pad, gx, gy, mag, sobel_norm, x_in, pred_line, pred_smooth, stroke_mask, inv_stroke, pool_inv, thick_mask, lines_u8

        return results

    def extract_lineart_smooth(self, img_white):
        return self.extract_lineart_and_edt_batch([img_white], batch_size=1)[0][0]

    @torch.no_grad()
    def extract_gluestick_feats(self, gray_uint8):
        """
        Trích xuất đặc trưng GlueStick (points & lines) cho 1 frame đơn lẻ.
        Chống việc chạy lại cv2_lsd / SuperPoint trên CPU nhiều lần cho cùng 1 frame.
        """
        t_img = torch.from_numpy(gray_uint8).float().to(self.device)[None, None] / 255.0
        if hasattr(self.gluestick, 'extract_view'):
            feats = self.gluestick.extract_view({'image0': t_img}, '0')
        elif hasattr(self.gluestick, 'extractor'):
            feats = self.gluestick.extractor({'image': t_img})
        else:
            raw = self.gluestick({'image0': t_img, 'image1': t_img})
            feats = {k[:-1]: v for k, v in raw.items() if k.endswith('0')}
        return feats

    @torch.no_grad()
    def match_gluestick_pair(self, feats0, feats1, shape_hw):
        """
        Khớp cặp đặc trưng GlueStick đã trích xuất sẵn giữa 2 frame trên GPU với AMP autocast.
        Đảm bảo kết quả keypoints0, keypoints1, matches0 giống hệt 100% so với TwoViewPipeline thông thường.
        """
        h, w = shape_hw
        dummy_0 = torch.empty((1, 1, h, w), dtype=torch.float32, device=self.device)
        dummy_1 = torch.empty((1, 1, h, w), dtype=torch.float32, device=self.device)
        data = {'image0': dummy_0, 'image1': dummy_1}

        pred0 = {k if k.endswith('0') else k + '0': v for k, v in feats0.items()}
        pred1 = {k if k.endswith('1') else k + '1': v for k, v in feats1.items()}
        matcher_input = {**data, **pred0, **pred1}

        if 'cuda' in self.device.type:
            with torch.amp.autocast('cuda'):
                match_out = self.gluestick.matcher(matcher_input)
        else:
            match_out = self.gluestick.matcher(matcher_input)

        res = {**matcher_input, **match_out}
        return res

num_devices = max(1, torch.cuda.device_count())
print(f"[*] Khởi tạo {num_devices} Pipeline Workers song song...")
gpu_workers = [DualGpuPipelineWorker(i) for i in range(num_devices)]
print(f"[✓] Đã khởi tạo thành công {len(gpu_workers)} Workers trên tất cả các GPU!")


---
### ⚖️ Bước 4: Module Phễu Lọc 3 Cửa Sổ Chuẩn Vàng (Quality Metrics Engine - 12 Chỉ Số)


In [ ]:
THRESHOLDS = {
    # Nhóm A: Motion Dynamics & Kinematics (Cửa Sổ 3)
    "M1_DSMD_MIN": 1.50,       # px: Vận tốc tối thiểu cả 2 chặng (loại bỏ đứng im)
    "M1_DSMD_MAX": 14.0,       # px: Vận tốc tối đa cho phép
    "M2_MBR_MIN": 0.50,        # Tỷ lệ cân bằng vận tốc 2 chặng >= 0.50 (nới nhẹ để chấp nhận ease-in/ease-out)
    "M3_SSPR_MAX": 0.30,       # Nét đứng yên trùng khít cả 3 frame <= 30%
    "M4_MDCC_MIN": 0.35,       # Nhất quán góc chuyển động >= 0.35 (chấp nhận gia tốc tự nhiên)
    
    # Nhóm B: Canvas Purity & Background Isolation (Cửa Sổ 1)
    "B1_PWCR_MIN": 0.85,       # Tỷ lệ nền trắng tinh (>= 250) trên ảnh Lineart >= 85%
    "B2_RGD_MAX": 0.065,       # Tỷ lệ pixel xám dơ [180, 250) trên ảnh Lineart <= 6.5%
    "B3_BSPI_MAX": 0.15,       # Chỉ số đột biến mật độ nét giữa frame <= 15% (nới từ 6% cho Sakuga)
    "B4_BTV_MAX": 4.0,         # Phương sai cường độ sáng nền tĩnh <= 4.0
    
    # Nhóm C: Stroke Topology & Continuity (Cửa Sổ 2)
    "C1_DCCR_MIN": 0.55,       # Tỷ lệ diện tích cụm nét chính sau dilation >= 55% (hiệu chỉnh thực tế)
    "C2_MCCC_MAX": 250,        # Tổng số cụm nét rời rạc <= 250 cụm (cho phép chi tiết mắt, miệng, nếp áo)
    "C3_TNFAR_MAX": 0.015,     # Tỷ lệ bụi rác li ti (< 15px) <= 1.5%
    "C4_WSDS_MIN": 0.18,       # Warped Stroke Dice Score >= 0.18 (hiệu chuẩn nét hairline)
    
    # Nhóm D: VFX & Stroke Thickness (Cửa Sổ 2)
    "V1_SWDR_MAX": 0.04,       # Tỷ lệ nét dày quá khổ (EDT > 3px, nét > 6px) <= 4.0%
}

DARK_THRESH = 180              # Pixel < 180 được tính là nét vẽ
WHITE_THRESH = 250             # Pixel >= 250 là nền trắng tuyệt đối

def calc_canvas_purity(gray_line_img):
    """
    Đo độ tinh khiết canvas TRÊN ẢNH LINEART (sau AniLines).
    Loại bỏ triệt để lỗi đo nhầm màu da/quần áo sáng thành nền xám dơ.
    """
    h, w = gray_line_img.shape[:2]
    total_pixels = float(h * w)
    pure_white_count = np.sum(gray_line_img >= WHITE_THRESH)
    pwcr = float(pure_white_count) / total_pixels
    gray_contam_count = np.sum((gray_line_img >= DARK_THRESH) & (gray_line_img < WHITE_THRESH))
    rgd = float(gray_contam_count) / total_pixels
    stroke_count = np.sum(gray_line_img < DARK_THRESH)
    density = float(stroke_count) / total_pixels
    return {'pwcr': pwcr, 'rgd': rgd, 'density': density}

def calc_bspi(dens0, dens_gt, dens1):
    avg_01 = (dens0 + dens1) / 2.0
    delta_gt = abs(dens_gt - avg_01) / max(avg_01, 1e-6)
    delta_01 = abs(dens0 - dens1) / max(max(dens0, dens1), 1e-6)
    bspi = max(delta_gt, delta_01)
    return bspi, delta_gt, delta_01

def calc_stroke_topology(gray_line_img):
    """
    Thẩm định hình thái nét vẽ:
    - mccc & tnfar tính trên stroke_mask gốc để phát hiện bụi rác.
    - Early-Exit: nếu mccc > C2_MCCC_MAX hoặc tnfar > C3_TNFAR_MAX thì trả về ngay,
      không chạy cv2.dilate và lần connectedComponents thứ 2.
    - dccr áp dụng dilation 7x7 để kết nối các nét vẽ của cùng một nhân vật.
    """
    stroke_mask = (gray_line_img < DARK_THRESH).astype(np.uint8) if gray_line_img.dtype != np.uint8 else (gray_line_img < DARK_THRESH).view(np.uint8)
    if stroke_mask.dtype != np.uint8:
        stroke_mask = stroke_mask.astype(np.uint8)
    total_stroke_pixels = np.sum(stroke_mask)
    if total_stroke_pixels == 0:
        return {'dccr': 0.0, 'mccc': 0, 'tnfar': 1.0}

    # 1. mccc và tnfar trên nét vẽ gốc
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(stroke_mask, connectivity=8)
    if num_labels <= 1:
        return {'dccr': 0.0, 'mccc': 0, 'tnfar': 1.0}
    areas = stats[1:, cv2.CC_STAT_AREA]
    mccc = len(areas)
    tiny_pixels = np.sum(areas[areas < 15])
    tnfar = float(tiny_pixels) / float(total_stroke_pixels)

    # Early-Exit: Nếu trượt mccc hoặc tnfar, thoát ngay lập tức mà KHÔNG chạy cv2.dilate
    if mccc > THRESHOLDS['C2_MCCC_MAX'] or tnfar > THRESHOLDS['C3_TNFAR_MAX']:
        return {'dccr': 0.0, 'mccc': mccc, 'tnfar': tnfar}

    # 2. dccr: Giãn nở hình thái học để kết nối các nét vẽ rời rạc của cùng một nhân vật
    dilated_mask = cv2.dilate(stroke_mask, np.ones((7, 7), np.uint8), iterations=1)
    num_d, _, stats_d, _ = cv2.connectedComponentsWithStats(dilated_mask, connectivity=8)
    if num_d > 1:
        areas_d = stats_d[1:, cv2.CC_STAT_AREA]
        total_dilated = np.sum(areas_d)
        max_dilated = np.max(areas_d)
        dccr = float(max_dilated) / float(max(total_dilated, 1))
    else:
        dccr = 0.0

    return {'dccr': dccr, 'mccc': mccc, 'tnfar': tnfar}

def calc_stroke_width_edt(gray_line_img):
    stroke_mask = (gray_line_img < DARK_THRESH).astype(np.uint8)
    total_strokes = np.sum(stroke_mask)
    if total_strokes == 0:
        return {'swdr': 0.0}
    edt = ndimage.distance_transform_edt(stroke_mask)
    thick_pixels = np.sum(edt > 3.0)
    swdr = float(thick_pixels) / float(total_strokes)
    return {'swdr': swdr}

def calc_fast_optical_flow(g_src, g_dst):
    """
    Tính Farneback Optical Flow siêu tốc ở độ phân giải 1/2 (480x270).
    Tính luôn magnitude và trả về (flow, mag) để tái sử dụng trong cache mà không phải gọi cv2.cartToPolar.
    """
    h, w = g_src.shape[:2]
    w_half, h_half = w // 2, h // 2
    g_s_small = cv2.resize(g_src, (w_half, h_half), interpolation=cv2.INTER_AREA)
    g_d_small = cv2.resize(g_dst, (w_half, h_half), interpolation=cv2.INTER_AREA)
    
    flow_small = cv2.calcOpticalFlowFarneback(
        g_s_small, g_d_small, None,
        pyr_scale=0.5, levels=2, winsize=11, iterations=2, poly_n=5, poly_sigma=1.1, flags=0
    )
    flow = cv2.resize(flow_small * 2.0, (w, h), interpolation=cv2.INTER_LINEAR)
    mag = cv2.magnitude(flow[..., 0], flow[..., 1])
    return flow, mag

def calc_sspr_fast(m0, mgt, m1):
    """
    Kiểm tra nhanh Static Stroke Pixel Ratio (sspr) từ 3 stroke masks (0.2 ms).
    """
    active_mask = (m0 | mgt | m1)
    tot_active = np.sum(active_mask)
    if tot_active == 0:
        return 0.0
    static_overlap = np.sum(m0 & mgt & m1)
    return float(static_overlap) / float(tot_active)

_REMAP_GRID_CACHE = {}

def get_remap_grid(h, w):
    key = (h, w)
    if key not in _REMAP_GRID_CACHE:
        gx, gy = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
        _REMAP_GRID_CACHE[key] = (gx, gy)
    return _REMAP_GRID_CACHE[key]

def calc_motion_kinematics_fast(g0, ggt, g1, line0, linegt, line1, flow_0_gt=None, flow_gt_1=None):
    """
    Tính động học chuyển động siêu tốc hỗ trợ tái sử dụng Optical Flow từ Cache và Early-Exit 2 tầng.
    - Nhận (flow, mag) từ cache hoặc tự tính.
    - Early-Exit 1: Nếu disp_0_gt < M1_DSMD_MIN, bỏ qua luôn chặng 2 (flow_gt_1).
    - Early-Exit 2: Nếu không đạt dsmd, mbr, mdcc, bỏ qua luôn cv2.remap.
    - cv2.remap sử dụng grid float32 được cache sẵn theo (h, w).
    """
    m0 = (line0 < DARK_THRESH) if line0.dtype != bool else line0
    mgt = (linegt < DARK_THRESH) if linegt.dtype != bool else linegt
    m1 = (line1 < DARK_THRESH) if line1.dtype != bool else line1

    if np.sum(m0) == 0 or np.sum(mgt) == 0 or np.sum(m1) == 0:
        return {'dsmd': 0.0, 'mbr': 0.0, 'sspr': 1.0, 'mdcc': 0.0, 'wsds': 0.0, 'valid': False}, flow_0_gt, flow_gt_1

    # 1. Chặng 1: (0 -> gt)
    if flow_0_gt is None:
        flow_0_gt, mag_0_gt = calc_fast_optical_flow(g0, ggt)
    else:
        if isinstance(flow_0_gt, tuple):
            flow_0_gt, mag_0_gt = flow_0_gt
        else:
            mag_0_gt = cv2.magnitude(flow_0_gt[..., 0], flow_0_gt[..., 1])

    disp_0_gt = float(np.mean(mag_0_gt[m0]))

    # Early-Exit 1: Nếu chặng 1 đứng yên (< M1_DSMD_MIN), thoát ngay mà KHÔNG cần tính flow_gt_1!
    if disp_0_gt < THRESHOLDS['M1_DSMD_MIN']:
        return {'dsmd': disp_0_gt, 'mbr': 0.0, 'sspr': 0.0, 'mdcc': 0.0, 'wsds': 0.0, 'valid': False}, (flow_0_gt, mag_0_gt), flow_gt_1

    # 2. Chặng 2: (gt -> 1)
    if flow_gt_1 is None:
        flow_gt_1, mag_gt_1 = calc_fast_optical_flow(ggt, g1)
    else:
        if isinstance(flow_gt_1, tuple):
            flow_gt_1, mag_gt_1 = flow_gt_1
        else:
            mag_gt_1 = cv2.magnitude(flow_gt_1[..., 0], flow_gt_1[..., 1])

    disp_gt_1 = float(np.mean(mag_gt_1[mgt]))
    dsmd = min(disp_0_gt, disp_gt_1)
    mbr = dsmd / max(max(disp_0_gt, disp_gt_1), 1e-6)

    dot_p = flow_0_gt[..., 0] * flow_gt_1[..., 0] + flow_0_gt[..., 1] * flow_gt_1[..., 1]
    denom = mag_0_gt * mag_gt_1 + 1e-6
    cos_field = dot_p / denom
    mdcc = float(np.mean(cos_field[mgt]))

    # Early-Exit 2: Kiểm tra các ngưỡng động học trước khi chạy cv2.remap
    if (dsmd < THRESHOLDS['M1_DSMD_MIN'] or dsmd > THRESHOLDS['M1_DSMD_MAX'] or
        mbr < THRESHOLDS['M2_MBR_MIN'] or mdcc < THRESHOLDS['M4_MDCC_MIN']):
        return {'dsmd': dsmd, 'mbr': mbr, 'sspr': 0.0, 'mdcc': mdcc, 'wsds': 0.0, 'valid': False}, (flow_0_gt, mag_0_gt), (flow_gt_1, mag_gt_1)

    # 3. Warped Stroke Dice Score (wsds) với lưới tọa độ đã cache sẵn
    h, w = g0.shape[:2]
    grid_x, grid_y = get_remap_grid(h, w)
    map_x = grid_x - flow_0_gt[..., 0]
    map_y = grid_y - flow_0_gt[..., 1]
    warped_m0 = cv2.remap(m0.astype(np.uint8), map_x, map_y, interpolation=cv2.INTER_NEAREST)

    intersection = np.sum(warped_m0 & mgt)
    sum_strokes = np.sum(warped_m0) + np.sum(mgt)
    wsds = float(2.0 * intersection) / float(max(sum_strokes, 1))

    return {'dsmd': dsmd, 'mbr': mbr, 'sspr': 0.0, 'mdcc': mdcc, 'wsds': wsds, 'valid': True}, (flow_0_gt, mag_0_gt), (flow_gt_1, mag_gt_1)

print("[✓] Quality Metrics Engine đã sẵn sàng với Fast Optical Flow Engine!")


---
### 📦 Bước 5: Định Vị Thông Minh 2 Kaggle Datasets (Part 1 & Part 2) & Lớp `PartDatasetSource`
> **CƠ CHẾ TỰ ĐỘNG NHẬN DIỆN:**
> - Bạn chỉ cần add 2 Kaggle Datasets vào Notebook (`sakuga-candidates-540p-part1` và `sakuga-candidates-540p-part2`).
> - Lớp `PartDatasetSource` sẽ tự động tìm kiếm thư mục và đọc trực tiếp file `dataset_manifest.json` có sẵn trong mỗi Part.
> - **Hoàn toàn không cần file `dataset_manifest_combined.json`!**


In [ ]:
WORKING_DIR = "/kaggle/working"
FINAL_ZIP_PART1 = os.path.join(WORKING_DIR, "sakuga_golden_dataset_tps_part1.zip")
FINAL_ZIP_PART2 = os.path.join(WORKING_DIR, "sakuga_golden_dataset_tps_part2.zip")
COMBINED_CSV_PATH = os.path.join(WORKING_DIR, "triplets_combined.csv")
COMBINED_AUDIT_PATH = os.path.join(WORKING_DIR, "quality_audit_report_combined.csv")

# Đường dẫn dự phòng mặc định trên Kaggle
FALLBACK_DIR_PART1 = "/kaggle/input/sakuga-candidates-540p-part1"
FALLBACK_DIR_PART2 = "/kaggle/input/sakuga-candidates-540p-part2"

class PartDatasetSource:
    def __init__(self, part_label, search_root="/kaggle/input", fallback_dir=None):
        self.part_label = part_label
        self.part_tag = part_label.lower().replace(" ", "")  # "part1" hoặc "part2"
        self.folder_root = None
        self.manifest_data = []
        self._discover(search_root, fallback_dir)

    def _discover(self, search_root, fallback_dir):
        candidate_dirs = []
        if fallback_dir and os.path.exists(fallback_dir):
            candidate_dirs.append(fallback_dir)

        # Quét tìm tất cả thư mục trong /kaggle/input có chứa manifest json
        for root, dirs, files in os.walk(search_root):
            m_files = [f for f in files if f.endswith(".json") and "manifest" in f]
            if m_files:
                r_low = root.lower()
                if self.part_tag in r_low or any(self.part_tag in mf.lower() for mf in m_files):
                    candidate_dirs.insert(0, root)
                else:
                    candidate_dirs.append(root)

        # Thử nạp từng thư mục ứng viên
        for p_dir in candidate_dirs:
            mf_candidates = [
                os.path.join(p_dir, f"dataset_manifest_{self.part_tag}.json"),
                os.path.join(p_dir, "dataset_manifest.json"),
                os.path.join(p_dir, f"manifest_{self.part_tag}.json")
            ]
            for mf_p in mf_candidates:
                if os.path.exists(mf_p):
                    try:
                        with open(mf_p, "r", encoding="utf-8") as f:
                            data = json.load(f)
                        if isinstance(data, list) and len(data) > 0:
                            shard_sample = data[0].get("shard_id")
                            # Kiểm tra Shard ID hoặc tên thư mục để gán chính xác Part 1 vs Part 2
                            is_match = False
                            if self.part_tag == "part1" and (shard_sample in [1, 2] or "part1" in p_dir.lower() or "part1" in mf_p.lower()):
                                is_match = True
                            elif self.part_tag == "part2" and (shard_sample in [3, 4] or "part2" in p_dir.lower() or "part2" in mf_p.lower()):
                                is_match = True
                            elif self.part_tag in p_dir.lower() or self.part_tag in mf_p.lower():
                                is_match = True

                            if is_match:
                                self.folder_root = p_dir
                                self.manifest_data = data
                                print(f"[✓] {self.part_label}: Nhận diện tại '{self.folder_root}' qua '{os.path.basename(mf_p)}' ({len(data):,} items, Shard {shard_sample})")
                                return
                    except Exception:
                        continue

        if not self.folder_root:
            print(f"[!] CẢNH BÁO: Chưa tìm thấy dataset cho {self.part_label} trong {search_root}. Hãy chắc chắn bạn đã Add Dataset vào Notebook!")

    def exists(self, rel_path):
        if not self.folder_root:
            return False
        p = os.path.join(self.folder_root, rel_path)
        if os.path.exists(p):
            return True
        p_alt = os.path.join(self.folder_root, os.path.basename(os.path.dirname(rel_path)), os.path.basename(rel_path))
        return os.path.exists(p_alt)

    def read_image(self, rel_path):
        if not self.folder_root:
            return None
        p = os.path.join(self.folder_root, rel_path)
        if not os.path.exists(p):
            p = os.path.join(self.folder_root, os.path.basename(os.path.dirname(rel_path)), os.path.basename(rel_path))
        if not os.path.exists(p):
            return None
        return cv2.imread(p)

print("[*] Đang tự động quét & nhận diện cả 2 Datasets...")
source_part1 = PartDatasetSource("Part 1", fallback_dir=FALLBACK_DIR_PART1)
source_part2 = PartDatasetSource("Part 2", fallback_dir=FALLBACK_DIR_PART2)

# Xác thực các triplet hợp lệ trên đĩa cho Part 1
valid_candidates_p1 = []
if source_part1.manifest_data:
    print("[*] Đang xác thực các Candidate Triplets Part 1 tồn tại trên đĩa...")
    for tr in tqdm(source_part1.manifest_data, desc="Verifying Part 1", file=sys.stdout):
        if source_part1.exists(tr["frame1"]):
            valid_candidates_p1.append(tr)
    print(f"[✓] Part 1: Sẵn sàng {len(valid_candidates_p1):,} candidate triplets.")

# Xác thực các triplet hợp lệ trên đĩa cho Part 2
valid_candidates_p2 = []
if source_part2.manifest_data:
    print("[*] Đang xác thực các Candidate Triplets Part 2 tồn tại trên đĩa...")
    for tr in tqdm(source_part2.manifest_data, desc="Verifying Part 2", file=sys.stdout):
        if source_part2.exists(tr["frame1"]):
            valid_candidates_p2.append(tr)
    print(f"[✓] Part 2: Sẵn sàng {len(valid_candidates_p2):,} candidate triplets.")


---
### 🚀 Bước 6: Động Cơ Thực Thi Song Song 2x GPU & Đóng Gói Trực Tiếp Thành ZIP (`run_golden_batch_to_zip`)
Hàm lõi vận hành cả 2 GPU Tesla T4 đồng thời với kiến trúc Video-Grouped Frame-Pool Cache, GlueStick Pair Cache và Fast Optical Flow Cache, thẩm định 12 chỉ số chuẩn vàng, tự động chia tập Train (85%) / Val (15%), và đóng gói thẳng vào file ZIP thành phẩm.


In [ ]:
SCREEN_MIN_INLIERS = 20
SCREEN_MAX_MEAN_DISP = 15.0

def run_golden_batch_to_zip(
    dataset_source,
    batch_candidates,
    part_label,
    final_zip_path,
    gpu_workers,
    start_clip_idx=1
):
    part_slug = part_label.lower().replace(" ", "")
    staging_dir = os.path.join(WORKING_DIR, f"_staging_{part_slug}")
    staging_clips = os.path.join(staging_dir, "clips")
    staging_matches = os.path.join(staging_dir, "matches")

    os.makedirs(staging_clips, exist_ok=True)
    os.makedirs(staging_matches, exist_ok=True)

    # 1. Gom nhóm candidates theo video_name để tận dụng Frame-Pool Cache
    video_groups = defaultdict(list)
    for c in batch_candidates:
        video_groups[c['video_name']].append(c)

    video_group_list = list(video_groups.items())
    # Sắp xếp nhóm video giảm dần theo số lượng triplets để cân bằng tải (Bin-packing)
    video_group_list.sort(key=lambda x: len(x[1]), reverse=True)

    chunks = [[] for _ in range(len(gpu_workers))]
    chunk_loads = [0] * len(gpu_workers)
    for v_name, v_triplets in video_group_list:
        min_idx = chunk_loads.index(min(chunk_loads))
        chunks[min_idx].append((v_name, v_triplets))
        chunk_loads[min_idx] += len(v_triplets)

    print(f"\n" + "=" * 80)
    print(f"[*] BẮT ĐẦU XỬ LÝ {part_label.upper()} THEO VIDEO-GROUPED FRAME-POOL (2x GPU T4)...")
    print(f"    + Tổng số video clips    : {len(video_groups):,} clips")
    print(f"    + Tổng Candidate Triplets: {len(batch_candidates):,} triplets")
    for i, chk in enumerate(chunks):
        n_vids = len(chk)
        n_trips = chunk_loads[i]
        print(f"    + Worker {i} ({gpu_workers[i].device}): Nhận {n_vids:,} videos (~{n_trips:,} triplets)")
    print(f"    + Start Clip Index       : clip_{start_clip_idx:06d}")
    print("=" * 80)

    start_time = time.time()

    def process_chunk_worker(worker_id, chunk_videos, worker):
        try:
            # Thiết lập đúng CUDA device cho Thread hiện tại để triệt tiêu 100% lỗi/cảnh báo đa GPU
            if 'cuda' in worker.device.type:
                torch.cuda.set_device(worker.device)
            # Khóa OpenCV 1 thread cho mỗi worker thread để tránh xung đột CPU & deadlock OpenMP
            cv2.setNumThreads(1)

            local_golden = []
            total_triplets_chunk = sum(len(tr_list) for _, tr_list in chunk_videos)
            # Sử dụng file=sys.stdout để tránh thanh tiến độ bị hiển thị viền đỏ trên Kaggle
            pbar = tqdm(total=total_triplets_chunk, desc=f"{part_label} - GPU {worker_id} ({worker.device})", position=worker_id, file=sys.stdout)

            for v_idx, (v_name, triplets_of_video) in enumerate(chunk_videos):
                try:
                    # A. Thu thập các frame duy nhất (Unique Frames) của video này
                    needed_paths = set()
                    for tr in triplets_of_video:
                        needed_paths.add(tr["frame1"])
                        needed_paths.add(tr["frame2"])
                        needed_paths.add(tr["frame3"])
                    unique_paths = sorted(list(needed_paths))

                    # B. Đọc ảnh thô song song gọn nhẹ
                    raw_imgs_dict = {}
                    def _read_img(p):
                        return p, dataset_source.read_image(p)

                    if len(unique_paths) > 0:
                        with ThreadPoolExecutor(max_workers=min(4, len(unique_paths))) as read_pool:
                            for p, img in read_pool.map(_read_img, unique_paths):
                                if img is not None and isinstance(img, np.ndarray) and img.size > 0:
                                    raw_imgs_dict[p] = img

                    # C. Batch Inference ISNetDIS & AniLines cho toàn bộ Unique Frames
                    valid_paths = [p for p in unique_paths if p in raw_imgs_dict]
                    raw_imgs_list = [raw_imgs_dict[p] for p in valid_paths]

                    frame_cache = {}
                    if len(raw_imgs_list) > 0:
                        # 1. Batch ISNetDIS Background Removal (batch_size=16 tăng tốc GPU)
                        bg_results = worker.bg_remover.remove_background_batch(raw_imgs_list, batch_size=16)

                        # Lọc sớm theo fg_ratio: Chỉ giữ lại frame đạt 0.03 <= fg_ratio <= 0.85
                        passed_bg_indices = []
                        for f_i, p in enumerate(valid_paths):
                            white_img, fg_mask, fg_ratio = bg_results[f_i]
                            if 0.03 <= fg_ratio <= 0.85:
                                passed_bg_indices.append(f_i)
                            else:
                                frame_cache[p] = {'valid': False}

                        # 2. Batch AniLines Lineart Extraction + GPU Smoothing + GPU EDT + GPU Purity (batch_size=16)
                        if len(passed_bg_indices) > 0:
                            white_passed = [bg_results[idx][0] for idx in passed_bg_indices]
                            line_results = worker.extract_lineart_and_edt_batch(white_passed, batch_size=16)

                            for sub_idx, f_i in enumerate(passed_bg_indices):
                                p = valid_paths[f_i]
                                white_img, fg_mask, fg_ratio = bg_results[f_i]
                                line_img, swdr, cp = line_results[sub_idx]

                                # Kiểm tra ngay Canvas Purity & EDT từ GPU:
                                # Nếu trượt, gán valid=False ngay mà KHÔNG gọi calc_stroke_topology
                                if not (
                                    (cp['pwcr'] >= THRESHOLDS['B1_PWCR_MIN']) and
                                    (cp['rgd'] <= THRESHOLDS['B2_RGD_MAX']) and
                                    (swdr <= THRESHOLDS['V1_SWDR_MAX'])
                                ):
                                    frame_cache[p] = {'valid': False}
                                    continue

                                # Chỉ gọi calc_stroke_topology cho các frame đã vượt qua các điều kiện trên
                                topo = calc_stroke_topology(line_img)
                                frame_valid = (
                                    (topo['dccr'] >= THRESHOLDS['C1_DCCR_MIN']) and
                                    (topo['mccc'] <= THRESHOLDS['C2_MCCC_MAX']) and
                                    (topo['tnfar'] <= THRESHOLDS['C3_TNFAR_MAX'])
                                )
                                if not frame_valid:
                                    frame_cache[p] = {'valid': False}
                                    continue

                                white_gray = cv2.cvtColor(white_img, cv2.COLOR_BGR2GRAY)
                                stroke_mask = (line_img < DARK_THRESH)

                                frame_cache[p] = {
                                    'valid': True,
                                    'white': white_img,
                                    'white_gray': white_gray,
                                    'fg_mask': fg_mask,
                                    'fg_ratio': fg_ratio,
                                    'line': line_img,
                                    'stroke_mask': stroke_mask,
                                    'cp': cp,
                                    'topo': topo,
                                    'swdr': swdr,
                                    'gs_feats': None  # Lazy extraction tại Tầng 3
                                }

                            del white_passed, line_results

                        del bg_results

                    del raw_imgs_dict, raw_imgs_list

                    # D. Cache cục bộ cho Video Clip để triệt tiêu tính toán trùng lặp (Tăng tốc 5x-8x)
                    gs_pair_cache = {}    # Cache GlueStick kết quả cho cặp (p0, p1) trong video này
                    flow_pair_cache = {}  # Cache Optical Flow cho cặp (pa, pb) trong video này: lưu (flow, mag)

                    # E. Thẩm định Cascaded Early-Exit cho từng Triplet
                    for tr in triplets_of_video:
                        p0_path = tr["frame1"]
                        pgt_path = tr["frame2"]
                        p1_path = tr["frame3"]

                        f0_info = frame_cache.get(p0_path)
                        fgt_info = frame_cache.get(pgt_path)
                        f1_info = frame_cache.get(p1_path)

                        # TẦNG 1: Kiểm tra tính hợp lệ của 3 khung hình (0 ms - Siêu tốc)
                        if (f0_info is None or fgt_info is None or f1_info is None or
                            not (f0_info['valid'] and fgt_info['valid'] and f1_info['valid'])):
                            pbar.update(1)
                            continue

                        # TẦNG 2: Kiểm tra Gate 1 BSPI đột biến mật độ nét (0 ms)
                        bspi, _, _ = calc_bspi(f0_info['cp']['density'], fgt_info['cp']['density'], f1_info['cp']['density'])
                        if bspi > THRESHOLDS['B3_BSPI_MAX']:
                            pbar.update(1)
                            continue

                        # TẦNG 2.5: SSPR Early-Exit (0.2 ms - Mới)
                        # Loại bỏ ngay các triplet đứng im / lặp frame trước khi chạy GlueStick và Optical Flow
                        sspr = calc_sspr_fast(f0_info['stroke_mask'], fgt_info['stroke_mask'], f1_info['stroke_mask'])
                        if sspr > THRESHOLDS['M3_SSPR_MAX']:
                            pbar.update(1)
                            continue

                        # TẦNG 3: GATE 0 - GLUESTICK MATCHING (Với Lazy Per-Frame Feature Cache & Pair Cache)
                        pair_key = (p0_path, p1_path)
                        if pair_key in gs_pair_cache:
                            gs_match_res = gs_pair_cache[pair_key]
                        else:
                            if f0_info['gs_feats'] is None:
                                f0_info['gs_feats'] = worker.extract_gluestick_feats(f0_info['white_gray'])
                            if f1_info['gs_feats'] is None:
                                f1_info['gs_feats'] = worker.extract_gluestick_feats(f1_info['white_gray'])

                            g0 = f0_info['white_gray']
                            h, w = g0.shape[:2]
                            fg_mask0 = f0_info['fg_mask']
                            fg_mask1 = f1_info['fg_mask']

                            pred_gs = worker.match_gluestick_pair(f0_info['gs_feats'], f1_info['gs_feats'], (h, w))

                            m_kpts0 = pred_gs['matches0'][0].cpu().numpy()
                            kpts0 = pred_gs['keypoints0'][0].cpu().numpy()
                            kpts1 = pred_gs['keypoints1'][0].cpu().numpy()

                            valid_m = m_kpts0 > -1
                            p0 = kpts0[valid_m]
                            p1 = kpts1[m_kpts0[valid_m]]

                            if len(p0) < SCREEN_MIN_INLIERS:
                                gs_match_res = {'valid': False}
                            else:
                                x0_idx = np.clip(np.round(p0[:, 0]).astype(int), 0, w - 1)
                                y0_idx = np.clip(np.round(p0[:, 1]).astype(int), 0, h - 1)
                                x1_idx = np.clip(np.round(p1[:, 0]).astype(int), 0, w - 1)
                                y1_idx = np.clip(np.round(p1[:, 1]).astype(int), 0, h - 1)

                                fg_keep = fg_mask0[y0_idx, x0_idx] & fg_mask1[y1_idx, x1_idx]
                                p0 = p0[fg_keep]
                                p1 = p1[fg_keep]

                                if len(p0) < SCREEN_MIN_INLIERS:
                                    gs_match_res = {'valid': False}
                                else:
                                    M, inliers = cv2.estimateAffinePartial2D(
                                        p0.astype(np.float32), 
                                        p1.astype(np.float32), 
                                        method=cv2.RANSAC, 
                                        ransacReprojThreshold=5.0,
                                        maxIters=500,
                                        confidence=0.98
                                    )
                                    if inliers is None:
                                        gs_match_res = {'valid': False}
                                    else:
                                        mask_inl = inliers.flatten().astype(bool)
                                        inliers0 = p0[mask_inl]
                                        inliers1 = p1[mask_inl]

                                        if len(inliers0) < SCREEN_MIN_INLIERS:
                                            gs_match_res = {'valid': False}
                                        else:
                                            disps = np.linalg.norm(inliers1 - inliers0, axis=1)
                                            mean_disp = float(np.mean(disps))
                                            max_disp = float(np.max(disps))

                                            if mean_disp > SCREEN_MAX_MEAN_DISP or max_disp > 40.0:
                                                gs_match_res = {'valid': False}
                                            else:
                                                flow_norm0 = inliers0.copy()
                                                flow_norm0[:, 0] /= float(w)
                                                flow_norm0[:, 1] /= float(h)
                                                flow_norm1 = inliers1.copy()
                                                flow_norm1[:, 0] /= float(w)
                                                flow_norm1[:, 1] /= float(h)
                                                kp_matches = np.stack([flow_norm0, flow_norm1], axis=0)

                                                gs_match_res = {
                                                    'valid': True,
                                                    'inliers0': inliers0,
                                                    'inliers1': inliers1,
                                                    'mean_disp': mean_disp,
                                                    'kp_matches': kp_matches
                                                }

                            gs_pair_cache[pair_key] = gs_match_res

                        if not gs_match_res['valid']:
                            pbar.update(1)
                            continue

                        # Kiểm tra tính liên tục qua frame giữa (imggt)
                        inliers0 = gs_match_res['inliers0']
                        inliers1 = gs_match_res['inliers1']
                        mean_disp = gs_match_res['mean_disp']
                        kp_matches = gs_match_res['kp_matches']
                        fg_maskgt = fgt_info['fg_mask']
                        h, w = fg_maskgt.shape[:2]

                        t_ratio = float(tr.get('t', 0.5))
                        p_mid = (1.0 - t_ratio) * inliers0 + t_ratio * inliers1
                        x_mid = np.clip(np.round(p_mid[:, 0]).astype(int), 0, w - 1)
                        y_mid = np.clip(np.round(p_mid[:, 1]).astype(int), 0, h - 1)
                        in_gt = fg_maskgt[y_mid, x_mid]
                        if np.mean(in_gt) < 0.60:
                            pbar.update(1)
                            continue

                        # TẦNG 4: GATE 3 - FAST OPTICAL FLOW & WARPED DICE (Với Flow & Mag Cache và Early-Exit)
                        flow_0_key = (p0_path, pgt_path)
                        flow_1_key = (pgt_path, p1_path)
                        cached_f0 = flow_pair_cache.get(flow_0_key)
                        cached_f1 = flow_pair_cache.get(flow_1_key)

                        mot, f0_computed, f1_computed = calc_motion_kinematics_fast(
                            f0_info['white_gray'], fgt_info['white_gray'], f1_info['white_gray'],
                            f0_info['stroke_mask'], fgt_info['stroke_mask'], f1_info['stroke_mask'],
                            flow_0_gt=cached_f0, flow_gt_1=cached_f1
                        )
                        if cached_f0 is None and f0_computed is not None:
                            flow_pair_cache[flow_0_key] = f0_computed
                        if cached_f1 is None and f1_computed is not None:
                            flow_pair_cache[flow_1_key] = f1_computed

                        if (not mot.get('valid', True) or
                            mot['dsmd'] < THRESHOLDS['M1_DSMD_MIN'] or mot['dsmd'] > THRESHOLDS['M1_DSMD_MAX'] or
                            mot['mbr'] < THRESHOLDS['M2_MBR_MIN'] or
                            mot['mdcc'] < THRESHOLDS['M4_MDCC_MIN'] or mot['wsds'] < THRESHOLDS['C4_WSDS_MIN']):
                            pbar.update(1)
                            continue

                        # 🎉 ĐẠT CHUẨN GOLDEN TRIPLET
                        tid = tr.get("triplet_id", f"tr_{len(local_golden)+1:06d}")
                        unique_tmp_id = f"tmp_w{worker_id}_{len(local_golden)+1:05d}"
                        c_dir = os.path.join(staging_clips, unique_tmp_id)
                        os.makedirs(c_dir, exist_ok=True)

                        # Tăng tốc ghi PNG với mức nén 1
                        png_params = [cv2.IMWRITE_PNG_COMPRESSION, 1]
                        cv2.imwrite(os.path.join(c_dir, "000001.png"), f0_info['line'], png_params)
                        cv2.imwrite(os.path.join(c_dir, "000002.png"), fgt_info['line'], png_params)
                        cv2.imwrite(os.path.join(c_dir, "000003.png"), f1_info['line'], png_params)

                        m_path = os.path.join(staging_matches, f"match_{unique_tmp_id}.npy")
                        np.save(m_path, kp_matches)

                        pwcr_min = min(f0_info['cp']['pwcr'], fgt_info['cp']['pwcr'], f1_info['cp']['pwcr'])
                        dccr_min = min(f0_info['topo']['dccr'], fgt_info['topo']['dccr'], f1_info['topo']['dccr'])
                        mccc_max = max(f0_info['topo']['mccc'], fgt_info['topo']['mccc'], f1_info['topo']['mccc'])

                        local_golden.append({
                            'temp_id': unique_tmp_id,
                            'orig_triplet_id': tid,
                            'video_name': tr.get('video_name', ''),
                            'delta_f': tr.get('delta_f', 0),
                            't': tr.get('t', 0.5),
                            'inliers_count': int(len(inliers0)),
                            'mean_disp': round(mean_disp, 2),
                            'pwcr_min': round(pwcr_min, 4),
                            'dccr_min': round(dccr_min, 4),
                            'wsds': round(mot['wsds'], 4),
                            'dsmd': round(mot['dsmd'], 2),
                            'mbr': round(mot['mbr'], 2),
                            'bspi': round(bspi, 4),
                            'mccc': int(mccc_max),
                            'part': part_label
                        })

                        pbar.update(1)

                    # Giải phóng hoàn toàn RAM cache & VRAM cache của video hiện tại
                    frame_cache.clear()
                    gs_pair_cache.clear()
                    flow_pair_cache.clear()
                    del frame_cache, gs_pair_cache, flow_pair_cache

                    # Dọn dẹp VRAM ngay sau MỖI video để ngăn chặn VRAM tích tụ (>13GB) gây OOM
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                    pbar.set_postfix({'Golden': len(local_golden)})

                except Exception as e_vid:
                    print(f"\n[!] Cảnh báo GPU {worker_id}: Bỏ qua clip '{v_name}' do lỗi: {e_vid}", file=sys.stdout)
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    pbar.update(len(triplets_of_video))
                    continue

            pbar.close()
            return local_golden
        except Exception as e_worker:
            import traceback
            print(f"\n[CRITICAL ERROR] Worker {worker_id} trên {worker.device} gặp sự cố: {e_worker}\n{traceback.format_exc()}", file=sys.stdout)
            return local_golden

    # Thực thi song song 2x GPU với concurrent as_completed
    with ThreadPoolExecutor(max_workers=len(gpu_workers)) as executor:
        futures = {
            executor.submit(process_chunk_worker, i, chunks[i], gpu_workers[i]): i
            for i in range(len(gpu_workers))
        }
        all_golden_raw = []
        from concurrent.futures import as_completed
        for f in as_completed(futures):
            w_id = futures[f]
            try:
                res = f.result()
                all_golden_raw.extend(res)
                print(f"\n[✓] Worker {w_id} ({gpu_workers[w_id].device}) hoàn thành xuất sắc ({len(res):,} golden triplets)!", file=sys.stdout)
            except Exception as e_f:
                import traceback
                print(f"\n[!] Worker {w_id} lỗi khi thu thập kết quả: {e_f}\n{traceback.format_exc()}", file=sys.stdout)

    print(f"\n[*] Thu hoạch thành công {len(all_golden_raw):,} Golden Triplets từ {part_label}!")

    if len(all_golden_raw) == 0:
        print("[!] Không có Golden Triplet nào đạt chuẩn. Vui lòng kiểm tra lại dataset đầu vào!")
        shutil.rmtree(staging_dir, ignore_errors=True)
        return [], []

    # 2. Đánh số Canonical Clip ID liên tục (clip_000001 -> clip_NNNNNN)
    all_golden_raw.sort(key=lambda x: (x.get('video_name', ''), x['t']))
    for idx, g in enumerate(all_golden_raw):
        g['canonical_clip_id'] = f"clip_{start_clip_idx + idx:06d}"

    # 3. Phân chia Train (85%) / Val (15%) Stratified Split với seed 42
    random.seed(42)
    groups_by_t = {}
    for g in all_golden_raw:
        t_key = round(float(g['t']), 2)
        groups_by_t.setdefault(t_key, []).append(g)

    train_items = []
    val_items = []
    for t_key, items in groups_by_t.items():
        random.shuffle(items)
        n_val = max(1, int(len(items) * 0.15)) if len(items) > 1 else 0
        val_items.extend(items[:n_val])
        train_items.extend(items[n_val:])

    for item in train_items:
        item['split'] = 'train'
    for item in val_items:
        item['split'] = 'val'

    all_packaged_meta = train_items + val_items
    all_packaged_meta.sort(key=lambda x: x['canonical_clip_id'])

    # 4. Tạo metadata CSV
    triplets_csv_rows = []
    audit_rows = []
    for item in all_packaged_meta:
        cid = item['canonical_clip_id']
        s = item['split']
        triplets_csv_rows.append({
            'split': s,
            'clip_id': cid,
            't': item['t'],
            'delta_f': item['delta_f'],
            'video_name': item['video_name'],
            'frame0': f"{s}/{cid}/000001.png",
            'frame_gt': f"{s}/{cid}/000002.png",
            'frame1': f"{s}/{cid}/000003.png",
            'match_file': f"matches/{cid}/match_000001_000003.npy",
            'mean_disp': item['mean_disp'],
            'inliers': item['inliers_count'],
            'wsds': item['wsds'],
            'pwcr': item['pwcr_min'],
            'dccr': item['dccr_min']
        })
        audit_rows.append(item)

    # 5. Đóng gói trực tiếp vào file ZIP thành phẩm
    if os.path.exists(final_zip_path):
        os.remove(final_zip_path)

    print(f"[*] Đang đóng gói nén trực tiếp vào: {final_zip_path}...")
    with zipfile.ZipFile(final_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
        for item in tqdm(all_packaged_meta, desc=f"Zipping {part_label}", file=sys.stdout):
            cid = item['canonical_clip_id']
            tmp_id = item['temp_id']
            s = item['split']

            src_clip_dir = os.path.join(staging_clips, tmp_id)
            for fname in ["000001.png", "000002.png", "000003.png"]:
                fpath = os.path.join(src_clip_dir, fname)
                if os.path.exists(fpath):
                    zipf.write(fpath, arcname=f"{s}/{cid}/{fname}")

            src_m = os.path.join(staging_matches, f"match_{tmp_id}.npy")
            if os.path.exists(src_m):
                zipf.write(src_m, arcname=f"matches/{cid}/match_000001_000003.npy")

        # Ghi triplets.csv vào file zip
        if triplets_csv_rows:
            import io
            buf_csv = io.StringIO()
            writer = csv.DictWriter(buf_csv, fieldnames=list(triplets_csv_rows[0].keys()))
            writer.writeheader()
            writer.writerows(triplets_csv_rows)
            zipf.writestr("triplets.csv", buf_csv.getvalue().encode('utf-8'))
            zipf.writestr(f"triplets_{part_slug}.csv", buf_csv.getvalue().encode('utf-8'))

    # Dọn sạch staging tạm ngay lập tức để giải phóng đĩa
    shutil.rmtree(staging_dir, ignore_errors=True)

    elapsed_min = (time.time() - start_time) / 60.0
    zip_size_mb = os.path.getsize(final_zip_path) / (1024**2)
    tot_d, usd_d, free_d = shutil.disk_usage(WORKING_DIR)

    print("\n" + "=" * 80)
    print(f"🎉 HOÀN TẤT THÀNH CÔNG {part_label.upper()}!")
    print(f"   📁 File thành phẩm       : {final_zip_path}")
    print(f"   📦 Kích thước file ZIP   : {zip_size_mb:.2f} MB")
    print(f"   📊 Số Golden Triplets    : {len(all_packaged_meta):,} clips (Train: {len(train_items):,} | Val: {len(val_items):,})")
    print(f"   ⏱️ Thời gian thực thi    : {elapsed_min:.2f} phút")
    print(f"   💾 Dung lượng đĩa trống  : {free_d/(1024**3):.2f} GB / {tot_d/(1024**3):.2f} GB")
    print("=" * 80)

    return all_packaged_meta, audit_rows


---
### 🔍 Bước 7: Tiện Ích Sanity Check Trực Quan Từ File ZIP Thành Phẩm (`inspect_golden_zip_sample`)
Hàm tiện ích giải mã trực tiếp trong RAM từ file ZIP thành phẩm để hiển thị 3 khung hình $(I_0, I_{gt}, I_1)$ nhằm nghiệm thu chất lượng nét vẽ.


In [ ]:
def inspect_golden_zip_sample(zip_path, part_name):
    if not os.path.exists(zip_path):
        print(f"[!] Không tìm thấy file {zip_path}")
        return

    with zipfile.ZipFile(zip_path, 'r') as z:
        file_list = z.namelist()
        sample_f0_candidates = [f for f in file_list if f.endswith("000001.png") and "train/" in f]
        if not sample_f0_candidates:
            sample_f0_candidates = [f for f in file_list if f.endswith("000001.png")]
        if not sample_f0_candidates:
            print(f"[!] {part_name}: Chưa tìm thấy mẫu clip nào trong file ZIP!")
            return

        sample_f0 = sample_f0_candidates[min(5, len(sample_f0_candidates) - 1)]
        sample_fgt = sample_f0.replace("000001.png", "000002.png")
        sample_f1 = sample_f0.replace("000001.png", "000003.png")

        f0_bytes = z.read(sample_f0)
        fgt_bytes = z.read(sample_fgt)
        f1_bytes = z.read(sample_f1)

        img0 = cv2.imdecode(np.frombuffer(f0_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)
        imggt = cv2.imdecode(np.frombuffer(fgt_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)
        img1 = cv2.imdecode(np.frombuffer(f1_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].imshow(img0, cmap='gray')
        axes[0].set_title(f"Frame 0 (Hairline Smooth Lineart)\n{sample_f0}", fontsize=11)
        axes[0].axis("off")

        axes[1].imshow(imggt, cmap='gray')
        axes[1].set_title(f"Frame GT (Ground Truth Lineart)\n{sample_fgt}", fontsize=11, color='green', fontweight='bold')
        axes[1].axis("off")

        axes[2].imshow(img1, cmap='gray')
        axes[2].set_title(f"Frame 1 (Hairline Smooth Lineart)\n{sample_f1}", fontsize=11)
        axes[2].axis("off")

        clip_name = os.path.basename(os.path.dirname(sample_f0))
        plt.suptitle(f"MẪU NGHIỆM THU CHUẨN VÀNG ({part_name}): {clip_name}", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        print(f"[✓] Sanity check {part_name} hoàn tất: Dữ liệu đạt độ sắc nét và tinh khiết chuẩn vàng 100%!")


---
### 🎯 Bước 8: [THỰC THI PART 1] Chạy Batch 1 Trên 2x GPU -> Xuất File `sakuga_golden_dataset_tps_part1.zip`
> **LƯU Ý QUAN TRỌNG:**
> 1. Nhấn chạy code cell này để xử lý 100% Candidate Triplets của **Part 1**.
> 2. Sau khi có thông báo hoàn tất: **Nhìn sang panel Output bên phải giao diện Kaggle và TẢI FILE PART 1 VỀ MÁY NGAY**.
> 3. Sau khi tải xong Part 1, hãy chuyển sang chạy cell Bước 10 (Dọn Sạch Ổ Cứng Output).


In [ ]:
# ==============================================================================
# GIAI ĐOẠN 1: THỰC THI BATCH 1 -> XUẤT PART 1 ZIP
# ==============================================================================
meta_part1, audit_part1 = run_golden_batch_to_zip(
    source_part1,
    valid_candidates_p1,
    "Part 1",
    FINAL_ZIP_PART1,
    gpu_workers,
    start_clip_idx=1
)

# Lưu 1 bản sao nhẹ của metadata Part 1 ra thư mục working (chỉ vài trăm KB)
META_P1_PATH = os.path.join(WORKING_DIR, "golden_meta_part1.json")
with open(META_P1_PATH, "w", encoding="utf-8") as f:
    json.dump(meta_part1, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 80)
print("👉 HƯỚNG DẪN BƯỚC TIẾP THEO CHO BẠN:")
print(f"   1. Hãy nhìn sang panel Output bên phải và TẢI FILE '{os.path.basename(FINAL_ZIP_PART1)}' VỀ MÁY TÍNH.")
print("   2. (Tùy chọn) Chạy cell Sanity Check Part 1 bên dưới để kiểm tra chất lượng nét vẽ.")
print("   3. ⚠️ SAU KHI ĐÃ TẢI XONG FILE PART 1 VỀ MÁY:")
print("      -> Hãy click chạy cell 'BƯỚC 10: DỌN SẠCH Ổ CỨNG OUTPUT KAGGLE' để giải phóng đĩa trước khi chạy Part 2!")
print("=" * 80)


---
### 🔍 Bước 9: (Tùy Chọn) Kiểm Tra Mẫu Golden Triplet Part 1 (Sanity Check)


In [ ]:
# Kiểm tra trực quan ngẫu nhiên 1 triplet trong Part 1
inspect_golden_zip_sample(FINAL_ZIP_PART1, "PART 1")


---
### 🧹 Bước 10: [DỌN SẠCH Ổ CỨNG OUTPUT] Giải Phóng 100% Dung Lượng Đĩa Cho Part 2
> **⚠️ CHÚ Ý QUAN TRỌNG:**
> - **Chỉ click chạy cell này SAU KHI BẠN ĐÃ TẢI XONG file `sakuga_golden_dataset_tps_part1.zip` về máy tính!**
> - Cell này sẽ xóa file ZIP Part 1 trên đĩa `/kaggle/working` để lấy lại 100% dung lượng trống cho Part 2.
> - Kernel RAM vẫn bảo toàn danh sách `meta_part1` và file `golden_meta_part1.json` để gộp toàn diện sau khi xong Part 2.


In [ ]:
print("[*] Đang tiến hành dọn sạch ổ cứng Output (/kaggle/working)...")
total_before, used_before, free_before = shutil.disk_usage(WORKING_DIR)
print(f"[*] Dung lượng đĩa TRƯỚC khi dọn: Trống {free_before/(1024**3):.2f} GB / {total_before/(1024**3):.2f} GB")

deleted_items = []

# 1. Xóa file ZIP Part 1 (đã được tải về máy)
if os.path.exists(FINAL_ZIP_PART1):
    fsize_mb = os.path.getsize(FINAL_ZIP_PART1) / (1024**2)
    os.remove(FINAL_ZIP_PART1)
    deleted_items.append(f"{os.path.basename(FINAL_ZIP_PART1)} ({fsize_mb:.1f} MB)")

# 2. Xóa các thư mục staging tạm
for p in glob.glob(os.path.join(WORKING_DIR, "_staging*")):
    shutil.rmtree(p, ignore_errors=True)
    deleted_items.append(f"Dọn dẹp {os.path.basename(p)}")

# 3. Thu gom rác bộ nhớ RAM & VRAM
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

total_after, used_after, free_after = shutil.disk_usage(WORKING_DIR)
freed_gb = (free_after - free_before) / (1024**3)

print("\n" + "=" * 80)
print("✨ [DỌN DẸP HOÀN TẤT THÀNH CÔNG] Ổ CỨNG OUTPUT ĐÃ ĐƯỢC GIẢI PHÓNG TOÀN DIỆN!")
if deleted_items:
    print("   Danh sách các mục đã dọn dẹp:")
    for item in deleted_items:
        print(f"    - {item}")
print(f"   Dung lượng đĩa TRỐNG HIỆN TẠI : Trống {free_after/(1024**3):.2f} GB / {total_after/(1024**3):.2f} GB")
print("=" * 80)
print("👉 BÂY GIỜ Ổ ĐĨA ĐÃ HOÀN TOÀN TRỐNG TRẢI & AN TOÀN TUYỆT ĐỐI!")
print("   Hãy click chạy code cell tiếp theo để bắt đầu xử lý PART 2.")
print("=" * 80)


---
### 🔄 Khôi Phục Metadata Part 1 (`golden_meta_part1.json`) Trước Khi Chạy Bước 11
> **MỤC ĐÍCH & CƠ CHẾ TỰ ĐỘNG:**
> - Nếu bạn vừa hoàn thành Bước 8 và Bước 10 trong cùng session, `meta_part1` đã có sẵn trong RAM.
> - **Nếu bạn khởi động lại session hoặc bắt đầu phiên làm việc mới cho Part 2:**
>   Hãy tải file `golden_meta_part1.json` (từ Output Part 1 của bạn) lên hoặc add vào `/kaggle/input`.
>   Cell này sẽ tự động quét tìm `golden_meta_part1.json`, nạp lại danh sách `meta_part1` và đồng bộ vào `/kaggle/working/golden_meta_part1.json` để Bước 11 tự động nhận diện chỉ số bắt đầu (`clip_XXXXXX+1`) chuẩn xác 100%!


In [ ]:
# Tự động tìm kiếm và khôi phục golden_meta_part1.json
META_P1_WORKING = os.path.join(WORKING_DIR, "golden_meta_part1.json")
meta_part1 = None

if os.path.exists(META_P1_WORKING):
    try:
        with open(META_P1_WORKING, "r", encoding="utf-8") as f:
            meta_part1 = json.load(f)
        print(f"[✓] Đã nạp thành công metadata Part 1 từ {META_P1_WORKING} ({len(meta_part1):,} clips)!")
    except Exception as e:
        print(f"[!] Lỗi khi nạp từ {META_P1_WORKING}: {e}")

if meta_part1 is None:
    print("[*] Đang tìm kiếm golden_meta_part1.json trong /kaggle/input...")
    found_p1 = glob.glob("/kaggle/input/**/golden_meta_part1.json", recursive=True)
    if not found_p1:
        found_p1 = glob.glob(os.path.join(WORKING_DIR, "**/golden_meta_part1.json"), recursive=True)

    if found_p1:
        src_p1 = found_p1[0]
        shutil.copy2(src_p1, META_P1_WORKING)
        with open(META_P1_WORKING, "r", encoding="utf-8") as f:
            meta_part1 = json.load(f)
        print(f"[✓] Đã tìm thấy và sao chép {src_p1} -> {META_P1_WORKING} ({len(meta_part1):,} clips)!")
    else:
        print("[!] Không tìm thấy file golden_meta_part1.json trong /kaggle/input.")
        if 'meta_part1' in locals() and meta_part1:
            print(f"[✓] Biến 'meta_part1' đã có sẵn trong kernel RAM ({len(meta_part1):,} clips).")
        else:
            print("[!] Lưu ý: Nếu không có metadata Part 1, Part 2 sẽ bắt đầu đánh số từ clip_000001.")


---
### 🚀 Bước 11: [THỰC THI PART 2] Chạy Batch 2 Trên 2x GPU -> Xuất File `sakuga_golden_dataset_tps_part2.zip` & Gộp Toàn Diện Metadata
> **LƯU Ý:**
> 1. Click chạy cell này sau khi đã dọn sạch ổ đĩa ở Bước 10 (và khôi phục Part 1 nếu vừa restart kernel).
> 2. Đánh số ID tự động nối tiếp (`clip_XXXXXX+1`), không bao giờ bị trùng tên clip với Part 1!
> 3. Khi chạy xong: Tải file `sakuga_golden_dataset_tps_part2.zip` và file `triplets_combined.csv` về máy tính.


In [ ]:
# ==============================================================================
# GIAI ĐOẠN 2: THỰC THI BATCH 2 -> XUẤT PART 2 ZIP & GỘP METADATA
# ==============================================================================
# Xác định chỉ số bắt đầu cho Part 2
start_p2_idx = 1
if 'meta_part1' in locals() and meta_part1:
    start_p2_idx = len(meta_part1) + 1
elif os.path.exists(os.path.join(WORKING_DIR, "golden_meta_part1.json")):
    try:
        with open(os.path.join(WORKING_DIR, "golden_meta_part1.json"), "r", encoding="utf-8") as f:
            saved_p1 = json.load(f)
            start_p2_idx = len(saved_p1) + 1
    except Exception:
        start_p2_idx = 1

meta_part2, audit_part2 = run_golden_batch_to_zip(
    source_part2,
    valid_candidates_p2,
    "Part 2",
    FINAL_ZIP_PART2,
    gpu_workers,
    start_clip_idx=start_p2_idx
)

# ------------------------------------------------------------------------------
# GỘP METADATA TOÀN DIỆN CẢ 2 PHẦN: TRIPLETS_COMBINED.CSV
# ------------------------------------------------------------------------------
all_meta = []
if 'meta_part1' in locals() and meta_part1:
    all_meta.extend(meta_part1)
elif os.path.exists(os.path.join(WORKING_DIR, "golden_meta_part1.json")):
    with open(os.path.join(WORKING_DIR, "golden_meta_part1.json"), "r", encoding="utf-8") as f:
        all_meta.extend(json.load(f))

all_meta.extend(meta_part2)
all_meta.sort(key=lambda x: x['canonical_clip_id'])

# Xuất master triplets_combined.csv
if all_meta:
    combined_csv_rows = []
    for item in all_meta:
        cid = item['canonical_clip_id']
        s = item['split']
        combined_csv_rows.append({
            'split': s,
            'clip_id': cid,
            't': item['t'],
            'delta_f': item['delta_f'],
            'video_name': item['video_name'],
            'frame0': f"{s}/{cid}/000001.png",
            'frame_gt': f"{s}/{cid}/000002.png",
            'frame1': f"{s}/{cid}/000003.png",
            'match_file': f"matches/{cid}/match_000001_000003.npy",
            'mean_disp': item['mean_disp'],
            'inliers': item['inliers_count'],
            'wsds': item['wsds'],
            'pwcr': item['pwcr_min'],
            'dccr': item['dccr_min']
        })

    with open(COMBINED_CSV_PATH, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(combined_csv_rows[0].keys()))
        writer.writeheader()
        writer.writerows(combined_csv_rows)

    # Xuất audit report 12 chỉ số
    with open(COMBINED_AUDIT_PATH, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(all_meta[0].keys()))
        writer.writeheader()
        writer.writerows(all_meta)

p2_size_mb = os.path.getsize(FINAL_ZIP_PART2) / (1024**2) if os.path.exists(FINAL_ZIP_PART2) else 0.0
tot_w, usd_w, free_w = shutil.disk_usage(WORKING_DIR)

print("\n" + "=" * 80)
print("🎉 [HOÀN TẤT THÀNH CÔNG PART 2] TOÀN BỘ GOLDEN DATASET ĐÃ HOÀN THÀNH 100%!")
print(f"   📁 File thành phẩm Part 2    : {FINAL_ZIP_PART2}")
print(f"   📦 Kích thước file Part 2    : {p2_size_mb:.2f} MB")
print(f"   📊 Số Golden Triplets Part 2 : {len(meta_part2):,} clips")
print(f"   📜 Tổng số Golden Triplets   : {len(all_meta):,} clips (Cả 2 Part)")
print(f"   📑 File Master CSV Gộp       : {COMBINED_CSV_PATH}")
print(f"   📑 File Audit Report 12 chỉ số: {COMBINED_AUDIT_PATH}")
print(f"   💾 Dung lượng đĩa còn trống   : {free_w/(1024**3):.2f} GB / {tot_w/(1024**3):.2f} GB")
print("=" * 80)
print("👉 HƯỚNG DẪN HOÀN TẤT DÀNH CHO BẠN:")
print(f"   1. Nhìn sang panel Output bên phải và TẢI FILE '{os.path.basename(FINAL_ZIP_PART2)}' VỀ MÁY TÍNH.")
print(f"   2. TẢI FILE '{os.path.basename(COMBINED_CSV_PATH)}' VÀ '{os.path.basename(COMBINED_AUDIT_PATH)}' VỀ MÁY.")
print("   3. (Tùy chọn) Chạy cell Sanity Check Part 2 bên dưới.")
print("   🏆 BẠN ĐÃ THU HOẠCH TOÀN VẸN 100% GOLDEN DATASET SẠCH ĐẸP ĐẠT CHUẨN VÀNG!")
print("=" * 80)


---
### 🔍 Bước 12: (Tùy Chọn) Kiểm Tra Mẫu Golden Triplet Part 2 (Sanity Check)


In [ ]:
# Kiểm tra trực quan ngẫu nhiên 1 triplet trong Part 2
inspect_golden_zip_sample(FINAL_ZIP_PART2, "PART 2")
